<h2>Description</h2>

Dans ce code, nous allons établir un modèle afin de prédire le débit horaire sur les Champs Élysées.

In [37]:
# =========================================================
# Recette "ancienne" adaptée : 2 modèles selon est_pieton
# =========================================================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# ---------- 0) Chargement & sécurisation ----------
df_final = pd.read_csv('../datasets_axes_with_all_features/champs_elysees.csv', sep=';')

if 'Date et heure de comptage' in df_final.columns:
    df_final = df_final.copy()
    df_final['Date et heure de comptage'] = pd.to_datetime(
        df_final['Date et heure de comptage'], errors='coerce'
    )
    df_final = df_final.sort_values('Date et heure de comptage').reset_index(drop=True)

# Cast indicatrices en numériques si besoin
for col in ['est_vacances', 'est_ferie', 'est_avant_ferie', 'est_pieton']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

# ---------- 1) Définition features / cible ----------
features = [
    'Température', 'precipitations heure', 'est_vacances', 'heure',
    'force moyenne vent (m/s)', 'mois', 'jour_semaine',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]
target = 'Débit horaire'

mask_known   = df_final[target].notna()
mask_missing = df_final[target].isna()

X_known = df_final.loc[mask_known, features].copy()
y_known = df_final.loc[mask_known, target].astype(float)
X_missing = df_final.loc[mask_missing, features].copy()

numeric_features = [
    'Température', 'precipitations heure', 'est_vacances', 'heure',
    'force moyenne vent (m/s)', 'mois', 'est_ferie', 'est_avant_ferie',
    'ensoleillement (en min)', 'est_pieton'
]
categorical_features = ['jour_semaine']

# ---------- 2) Prétraitements ----------
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features),
    ],
    remainder='drop'
)

def build_pipeline():
    model = HistGradientBoostingRegressor(
        loss='absolute_error',   # MAE : robuste aux valeurs extrêmes
        max_depth=4,
        max_iter=400,
        early_stopping=False,
        random_state=42
    )
    return Pipeline(steps=[('prep', preprocess), ('model', model)])

# ---------- 3) Split chronologique global ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_known, y_known, test_size=0.2, shuffle=False
)

# ---------- 4) Pondérations d'apprentissage (férié / veille) ----------
W_FERIE = 3.0
W_AVANT = 1.5
POST_SCALE_FERIE = 1.0  # conservez 1.0 si vous ne souhaitez pas corriger y_pred les jours fériés

def make_weights(X_frame):
    w = np.ones(len(X_frame), dtype=float)
    is_ferie = X_frame['est_ferie'].fillna(0).astype(int).to_numpy()
    is_avant = X_frame['est_avant_ferie'].fillna(0).astype(int).to_numpy()
    w[is_ferie == 1] = W_FERIE
    w[is_avant == 1] = np.maximum(w[is_avant == 1], W_AVANT)
    # Normalisation (optionnelle mais propre)
    w *= (len(w) / w.sum())
    return w

w_train_global = make_weights(X_train)

# ---------- 5) Entraînement par régime (est_pieton) ----------
MIN_REGIME_TRAIN = 200  # repli si < 200 obs. dans un régime

reg_train = X_train['est_pieton'].fillna(0).astype(int).to_numpy()
reg_test  = X_test['est_pieton'].fillna(0).astype(int).to_numpy()

idx_train_p  = np.where(reg_train == 1)[0]
idx_train_np = np.where(reg_train == 0)[0]
idx_test_p   = np.where(reg_test == 1)[0]
idx_test_np  = np.where(reg_test == 0)[0]

pipe_pieton = None
pipe_normal = None

# Modèle piéton
if len(idx_train_p) >= MIN_REGIME_TRAIN:
    pipe_pieton = build_pipeline()
    w_train_p = w_train_global[idx_train_p]
    pipe_pieton.fit(X_train.iloc[idx_train_p], y_train.iloc[idx_train_p],
                    model__sample_weight=w_train_p)

# Modèle non piéton
if len(idx_train_np) >= MIN_REGIME_TRAIN:
    pipe_normal = build_pipeline()
    w_train_np = w_train_global[idx_train_np]
    pipe_normal.fit(X_train.iloc[idx_train_np], y_train.iloc[idx_train_np],
                    model__sample_weight=w_train_np)

# Repli : modèle global si l'un des deux régimes est trop faible
pipe_global = None
if (pipe_pieton is None) or (pipe_normal is None):
    pipe_global = build_pipeline()
    pipe_global.fit(X_train, y_train, model__sample_weight=w_train_global)

# ---------- 6) Prédictions (routage par régime) ----------
y_pred = np.empty_like(y_test, dtype=float)

# Piéton
if len(idx_test_p) > 0:
    if pipe_pieton is not None:
        y_pred[idx_test_p] = pipe_pieton.predict(X_test.iloc[idx_test_p])
    else:
        y_pred[idx_test_p] = pipe_global.predict(X_test.iloc[idx_test_p])

# Non piéton
if len(idx_test_np) > 0:
    if pipe_normal is not None:
        y_pred[idx_test_np] = pipe_normal.predict(X_test.iloc[idx_test_np])
    else:
        y_pred[idx_test_np] = pipe_global.predict(X_test.iloc[idx_test_np])

# Post-ajustement fériés (optionnel)
mask_ferie_test = X_test['est_ferie'].fillna(0).to_numpy().astype(int) == 1
y_pred[mask_ferie_test] *= POST_SCALE_FERIE

# ---------- 7) Évaluation ----------
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² (global)  : {r2:.3f}")
print(f"MAE (global) : {mae:.2f}")
print(f"RMSE (global): {rmse:.2f}")

# Diagnostics par régimes piéton / non piéton
is_pieton_test = X_test['est_pieton'].fillna(0).astype(int) == 1
is_non_pieton_test = ~is_pieton_test

if is_pieton_test.any():
    mae_pieton = mean_absolute_error(y_test[is_pieton_test], y_pred[is_pieton_test])
    print(f"MAE (jours piéton) : {mae_pieton:.2f} (n={is_pieton_test.sum()})")
else:
    print("Aucun jour piéton dans le jeu de test.")

if is_non_pieton_test.any():
    mae_non_pieton = mean_absolute_error(y_test[is_non_pieton_test], y_pred[is_non_pieton_test])
    print(f"MAE (jours non piéton) : {mae_non_pieton:.2f} (n={is_non_pieton_test.sum()})")

# ---------- 8) Imputation des valeurs manquantes (routage idem) ----------
if mask_missing.any():
    X_miss = X_missing.copy()
    reg_miss = X_miss['est_pieton'].fillna(0).astype(int).to_numpy()

    y_miss_pred = np.empty(len(X_miss), dtype=float)
    idx_miss_p  = np.where(reg_miss == 1)[0]
    idx_miss_np = np.where(reg_miss == 0)[0]

    # Piéton
    if len(idx_miss_p) > 0:
        if pipe_pieton is not None:
            y_miss_pred[idx_miss_p] = pipe_pieton.predict(X_miss.iloc[idx_miss_p])
        elif pipe_global is not None:
            y_miss_pred[idx_miss_p] = pipe_global.predict(X_miss.iloc[idx_miss_p])

    # Non piéton
    if len(idx_miss_np) > 0:
        if pipe_normal is not None:
            y_miss_pred[idx_miss_np] = pipe_normal.predict(X_miss.iloc[idx_miss_np])
        elif pipe_global is not None:
            y_miss_pred[idx_miss_np] = pipe_global.predict(X_miss.iloc[idx_miss_np])

    df_final.loc[mask_missing, 'Débit_prédit'] = y_miss_pred
    print(f"Imputation réalisée pour {mask_missing.sum()} lignes (colonne 'Débit_prédit').")


R² (global)  : 0.758
MAE (global) : 89.79
RMSE (global): 124.87
MAE (jours piéton) : 218.64 (n=14)
MAE (jours non piéton) : 88.75 (n=1727)
Imputation réalisée pour 563 lignes (colonne 'Débit_prédit').


In [38]:
import plotly.graph_objects as go

time_index = df_final.loc[X_test.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_pred,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_test,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

In [12]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 0) Prédictions sur le jeu de test
y_pred = xgb.predict(X_val_t)

# 1) Reconstitution du DataFrame de diagnostic
n_test = len(y_test)
# récupère les n dernières dates (équivalent à df.loc[split:, 'Date et heure de comptage'])
dates_test = df['Date et heure de comptage'].iloc[-n_test:].values

diag = pd.DataFrame({
    'datetime': pd.to_datetime(dates_test),
    'y_true':   np.asarray(y_test),
    'y_pred':   np.asarray(y_pred)
}).sort_values('datetime').reset_index(drop=True)

diag['residual'] = diag['y_true'] - diag['y_pred']

# 2) Métriques globales
rmse = np.sqrt(mean_squared_error(diag['y_true'], diag['y_pred']))
mae  = mean_absolute_error(diag['y_true'], diag['y_pred'])
print(f"RMSE global : {rmse:,.2f}")
print(f"MAE  global : {mae:,.2f}")

# 3) Série temporelle Observé vs. Prédit
fig_ts = go.Figure()
fig_ts.add_trace(go.Scatter(x=diag['datetime'], y=diag['y_true'], mode='lines', name='Observé'))
fig_ts.add_trace(go.Scatter(x=diag['datetime'], y=diag['y_pred'], mode='lines', name='Prédit'))
fig_ts.update_layout(
    title="Débit horaire — Observé vs. Prédit (jeu de test)",
    xaxis_title="Date",
    yaxis_title="Débit horaire",
    legend_title_text="Série"
)
fig_ts.show()

# 4) Nuage de parité (y_true vs y_pred) + diagonale
min_v = float(np.min([diag['y_true'].min(), diag['y_pred'].min()]))
max_v = float(np.max([diag['y_true'].max(), diag['y_pred'].max()]))

fig_parity = go.Figure()
fig_parity.add_trace(go.Scatter(x=diag['y_true'], y=diag['y_pred'], mode='markers', name='Points', opacity=0.6))
fig_parity.add_trace(go.Scatter(x=[min_v, max_v], y=[min_v, max_v], mode='lines', name='y = x'))
fig_parity.update_layout(
    title="Parité — Observé vs. Prédit",
    xaxis_title="y_observé",
    yaxis_title="y_prédit"
)
fig_parity.show()

# 5) Résidus dans le temps
fig_res_time = go.Figure()
fig_res_time.add_trace(go.Scatter(x=diag['datetime'], y=diag['residual'], mode='lines', name='Résidu'))
fig_res_time.add_hline(y=0)
fig_res_time.update_layout(
    title="Résidus (y_observé − y_prédit) dans le temps",
    xaxis_title="Date",
    yaxis_title="Résidu"
)
fig_res_time.show()

# 6) Histogramme des résidus
fig_hist = px.histogram(diag, x='residual', nbins=40, title='Distribution des résidus')
fig_hist.update_layout(xaxis_title="Résidu", yaxis_title="Fréquence")
fig_hist.show()

# 7) RMSE glissant
window = 48  # ≈ 2 jours si série horaire sans trous
diag['se'] = (diag['y_true'] - diag['y_pred'])**2
diag['rmse_roll'] = np.sqrt(diag['se'].rolling(window=window, min_periods=window//2).mean())

fig_rmse_roll = go.Figure()
fig_rmse_roll.add_trace(go.Scatter(x=diag['datetime'], y=diag['rmse_roll'], mode='lines', name='RMSE glissant'))
fig_rmse_roll.update_layout(
    title=f"RMSE glissant (fenêtre = {window})",
    xaxis_title="Date",
    yaxis_title="RMSE"
)
fig_rmse_roll.show()

# 8) MAE par heure de la journée
tmp = diag.copy()
tmp['heure'] = pd.to_datetime(tmp['datetime']).dt.hour
mae_par_heure = tmp.groupby('heure').apply(lambda z: mean_absolute_error(z['y_true'], z['y_pred'])).reset_index(name='MAE')

fig_mae_hour = px.bar(mae_par_heure, x='heure', y='MAE', title='MAE par heure de la journée')
fig_mae_hour.update_layout(xaxis_title="Heure", yaxis_title="MAE")
fig_mae_hour.show()


RMSE global : 165.93
MAE  global : 125.45


/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_45149/31949715.py:88: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

